In [ ]:
from sqlalchemy.testing.suite.test_reflection import metadata
%pip install uv
# Qdrant의 Sparse Vector 생성을 위해 fastembed 설치 필요
%pip install fastembed

In [ ]:
# !uv pip install -U langchain langchain-openai qdrant-client langchain-qdrant langgraph langchain-upstage langchain-community langchain-teddynote pip-chill python-dotenv mysql-connector-python fastembed

In [ ]:
from qdrant_client.http.models import VectorParams, Distance
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from langchain_upstage import UpstageEmbeddings
from langchain_core.documents import Document
from qdrant_client import QdrantClient, models
import mysql.connector
import os
from dotenv import find_dotenv, load_dotenv

# 1. 환경 변수 로드
load_dotenv(find_dotenv())

# 2. DB 설정
db_config = {
    'host': os.getenv('DB_HOST'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'database': os.getenv('DB_NAME')
}

# 3. 배치 설정
BATCH_SIZE = 400
# 중요: Dense + Sparse 벡터를 모두 저장하려면 컬렉션 설정이 달라져야 하므로 이름을 변경하거나 기존 컬렉션을 삭제 후 재생성해야 합니다.
collection_name = "attractions_hybrid" 
qdrant_url = os.getenv('QDRANT_URL')

# 4. 모델 및 클라이언트 초기화
embedding_model = UpstageEmbeddings(model="solar-embedding-1-large-passage")
dimension = len(embedding_model.embed_query("check"))

# Sparse Embedding 모델 (BM25와 유사한 역할을 하는 Sparse Vector 생성)
# Qdrant/bm25 모델이 다국어 토큰화 BM25를 지원합니다.
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

# Qdrant 클라이언트 (메모리 모드)
client = QdrantClient(":memory:")

# 컬렉션 생성 (QdrantVectorStore가 자동으로 해주지만, 설정을 명확히 하기 위해 직접 생성)
# LangChain QdrantVectorStore는 sparse_embedding을 전달하면 자동으로 sparse_vector config를 추가해줍니다.
# 따라서 여기서는 클라이언트 초기화만 하고 생성은 LangChain에게 맡기거나,
# vector_store.exist 메서드 체크 없이 바로 연결합니다.

# 5. LangChain Qdrant 저장소 연결 (Hybrid 모드)
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embedding_model,
    sparse_embedding=sparse_embeddings, # Sparse 모델 추가
    retrieval_mode=RetrievalMode.HYBRID # 하이브리드 검색 모드 활성화
)

In [ ]:
try:
    connection = mysql.connector.connect(**db_config)
    cursor = connection.cursor(dictionary=True)

    offset = 0
    total_inserted = 0

    print("--- 배치 처리 시작 ---")

    for _ in range(1):  # 테스트를 위해 1회만 실행
        query = f"""
            SELECT *
            FROM attractions
            WHERE overview IS NOT NULL AND overview != ''
            LIMIT {BATCH_SIZE} OFFSET {offset}
        """

        cursor.execute(query)
        rows = cursor.fetchall()

        if not rows:
            print("--- 모든 데이터 처리 완료 ---")
            break

        documents = [Document(
            page_content=row["overview"],
            metadata={
                "location": {
                    "lat": float(row["latitude"]),
                    "lon": float(row["longitude"])
                },
                # 필터링 테스트를 위해 지역 코드 등 추가 메타데이터 확보
                "area_code": row.get("area_code", 0),
                **{k: v for k, v in row.items() if k not in ["overview", "latitude", "longitude"]}
            }
        ) for row in rows if float(row["latitude"]) != 0 and float(row["longitude"]) != 0]

        if documents:
            # 데이터 저장 (Dense 벡터와 Sparse 벡터가 함께 생성되어 저장됨)
            vector_store.add_documents(documents)
            
            total_inserted += len(documents)
            print(f"Current Batch: {len(documents)}건 저장 (누적: {total_inserted}건) | Offset: {offset}")

        offset += BATCH_SIZE

except mysql.connector.Error as err:
    print(f"Error: {err}")
finally:
    if 'connection' in locals() and connection.is_connected():
        cursor.close()
        connection.close()
        print("MySQL 연결 종료")

In [ ]:
# 6. 위치 기반 필터링 + 하이브리드 검색 테스트

# Qdrant 모델 필터 정의 (예: 특정 반경 내 검색)
# 참고: Qdrant는 Geo Radius 필터링을 지원합니다.
from qdrant_client.http import models as rest_models

query = "성당" # 키워드 (Sparse가 잘 잡아야 함)

# 예시: 서울 종로구 북촌로 근처 (37.58, 126.98) 반경 2km 이내 필터
lat, lon = 37.58, 126.98
radius_km = 2.0

geo_filter = rest_models.Filter(
    must=[
        rest_models.FieldCondition(
            key="metadata.location",
            geo_radius=rest_models.GeoRadius(
                center=rest_models.GeoPoint(
                    lat=lat,
                    lon=lon
                ),
                radius=radius_km * 1000.0  # 미터 단위
            )
        )
    ]
)

print(f"검색어: '{query}' | 필터: 반경 {radius_km}km 이내")

# similarity_search 호출 시 filter 전달 -> Dense/Sparse 모두에 적용됨
results = vector_store.similarity_search(
    query,
    k=3,
    filter=geo_filter
)

for i, doc in enumerate(results):
    print(f"[{i+1}] {doc.metadata.get('title', 'No Title')}")
    print(f"   위치: {doc.metadata.get('location')}")
    print(f"   내용: {doc.page_content[:100]}...")
    print("-" * 50)